### For retreival stage pinecone similarity socre is imp . For Pinecone similarity score — is higher better
### For rerank stage rerank score is more imp . rerank score higher is better

In [1]:
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_cohere import CohereEmbeddings, CohereRerank, ChatCohere
from langchain_pinecone import PineconeVectorStore
from langchain_openai import ChatOpenAI
from langchain_community.retrievers import PineconeHybridSearchRetriever
from langchain_openai import OpenAIEmbeddings
from pinecone import Pinecone, ServerlessSpec
import cohere
from IPython.display import display, Markdown



from dotenv import load_dotenv
load_dotenv()

os.environ["COHERE_API_KEY"] = os.getenv("COHERE_API_KEY")
os.environ["PINECONE_API_KEY"] = os.getenv("PINECONE_API_KEY")


/Users/ashishtak/GenAI/genv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## VectorDB (Pinecone)

In [2]:
pc = Pinecone()

index_name = "rerank-hybrid-demo"

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=1536,  #  embedding model dimension
        metric="dotproduct",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)
index

# OpenAI Embeddings

In [3]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [4]:
# checking dimenisons
vector = embeddings.embed_query("xyz")
print(len(vector))

1536


## Cleaning and Chuking

In [5]:
import re

# 1️⃣ Load PDF
loader = PyPDFLoader("/Users/ashishtak/GenAI/3-Vector-DB/king.dreamspeech.excerpts.pdf")
docs = loader.load()

# 2️⃣ Clean Text Function
def clean_text(text: str) -> str:
    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)
    
    # Remove copyright lines
    text = re.sub(r'©.*?\n', '', text)
    
    # Remove Gilder Lehrman headers
    text = re.sub(r'The Gilder Lehrman Institute.*?\n', '', text)
    
    # Remove excessive whitespace
    text = re.sub(r'\n{2,}', '\n', text)
    
    return text.strip()

# 3️⃣ Apply Cleaning
cleaned_docs = []
for doc in docs:
    cleaned_content = clean_text(doc.page_content)
    cleaned_docs.append(
        Document(
            page_content=cleaned_content,
            metadata=doc.metadata  # preserve page numbers
        )
    )

# 4️⃣ Better Chunking for Speeches
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,      # smaller chunks = more precise retrieval
    chunk_overlap=80     # slightly more overlap for continuity
)

split_docs = splitter.split_documents(cleaned_docs)

print(f"Total chunks created: {len(split_docs)}")

Total chunks created: 18


## VectorDB

In [6]:
vectorstore = PineconeVectorStore(
    index=index,
    embedding=embeddings,
    text_key="context" # this name should be same in the retriever
)

vectorstore.add_documents(split_docs)


['271fe2c1-0c8d-48ad-b0dc-2976453092f7',
 'd2a81c48-6583-49c8-9104-05253de9e166',
 '734005d5-42a4-4bef-9bee-4e56d88c998e',
 'c8549fa0-48dc-4525-aba4-5df898f25eaf',
 'b6fe92cb-332c-4653-b00f-e2464f186378',
 '2ae2ed3b-4e4b-42ac-aa2e-efeeac174953',
 '1ed0dc00-41ed-40e2-ac01-658e2eaa01f5',
 '6d4b806a-3f31-48d6-a7e7-a01437ce0c2e',
 '9c597906-2744-42b0-854b-8b60a8420e3b',
 '95ebc78b-08d1-491e-8b91-a772d2dbccb0',
 '6049b2ab-e400-4d38-a9f5-c57eae624f4c',
 '236a9438-e412-4042-8bc3-b97ec4653de0',
 '5b447fbc-8382-4e7a-ba6f-83240264e967',
 'b11b4a3c-04f2-4817-8b43-88eb2fa8d82e',
 'dd5282a2-6b83-4f4c-8f7d-b05e81a75d47',
 '9238908d-e0f0-4b91-90bb-873623a76983',
 '791b9b59-ff4c-45b8-8e7d-3f149c6020e2',
 'bc2f7a2b-a613-4df3-95f4-10096711b08c']

## BM25-Encoder

In [7]:
from pinecone_text.sparse import BM25Encoder # use tfidf bydefault
bm25_encoder = BM25Encoder().default()
bm25_encoder

bm25_encoder.fit([doc.page_content for doc in split_docs])## very imp

bm25_encoder.dump("Rerank_bm25.json")
bm25_encoder = BM25Encoder().load("Rerank_bm25.json")

100%|██████████| 18/18 [00:00<00:00, 136.70it/s]


## Retriever (Hybrid Search)

In [8]:
hybrid_retriever = PineconeHybridSearchRetriever(
    index=index,
    embeddings=embeddings,
    sparse_encoder=bm25_encoder,
    text_key="context",   # must match your metadata key while we made the embeedings
    alpha=0.8,
    top_k=5
)

# Cohere Rerank Model

In [9]:
reranker = CohereRerank(model="rerank-v4.0-pro") # rerank-english-v3.0

## Before Rerank (with pinecone similarity score)

In [10]:
query = "What was Martin Luther King Jr.s dream about racial equality?"
docs = hybrid_retriever.invoke(query)

for i, doc in enumerate(docs, 1):
    print(f"Rank {i}")
    print("Score:", doc.metadata.get("score"))
    print(doc.page_content[:500])
    print("-" * 80)




Rank 1
Score: 0.419176102
“I Have a Dream” Speech by the Rev. Martin Luther King Jr. at the “March on Washington,” 
1963 (excerpts ) 
 
I am happy to join with you today in what will go down in history as the greatest demonstration for 
freedom in the history of our nation. 
Five score years ago a great American in whose symbolic shadow we stand today signed the
--------------------------------------------------------------------------------
Rank 2
Score: 0.419176102
“I Have a Dream” Speech by the Rev. Martin Luther King Jr. at the “March on Washington,” 
1963 (excerpts ) 
 
I am happy to join with you today in what will go down in history as the greatest demonstration for 
freedom in the history of our nation. 
Five score years ago a great American in whose symbolic shadow we stand today signed the
--------------------------------------------------------------------------------
Rank 3
Score: 0.419116974
“I Have a Dream” Speech by the Rev. Martin Luther King Jr. at the “March on Washin

## After Rerank (with pinecone similarity score)

```bash 
rerank returns = 
{  "results": [
    {
      "index": 3,
      "relevance_score": 0.999071
    },
    {
      "index": 4,
      "relevance_score": 0.7867867
    },
    {
      "index": 0,
      "relevance_score": 0.32713068
    }],
}

In [11]:
reranked_docs = reranker.compress_documents(docs, query)
for i, doc in enumerate(reranked_docs, 1):
    print(f"Rank {i}")
    print("Score:", doc.metadata.get("score"))
    print(doc.page_content[:500])
    print("-" * 80)


Rank 1
Score: 0.403454781
the color of their skin but by the content of their character. I have a dream . . . I have a dream that one 
day in Alabama, with its vicious racists, with its governor having his lips dripping with the words of 
interposition and nullification, one day right there in Alabama little black boys and black girls will be able
--------------------------------------------------------------------------------
Rank 2
Score: 0.419176102
“I Have a Dream” Speech by the Rev. Martin Luther King Jr. at the “March on Washington,” 
1963 (excerpts ) 
 
I am happy to join with you today in what will go down in history as the greatest demonstration for 
freedom in the history of our nation. 
Five score years ago a great American in whose symbolic shadow we stand today signed the
--------------------------------------------------------------------------------
Rank 3
Score: 0.419176102
“I Have a Dream” Speech by the Rev. Martin Luther King Jr. at the “March on Washington,” 
1963 (e

# Reranking Socre

In [12]:
query = "What was Martin Luther King Jr.'s dream about racial equality?"
docs = hybrid_retriever.invoke(query)

doc_texts = [
    doc.page_content.strip()
    for doc in docs
    if doc.page_content and doc.page_content.strip()
]


co = cohere.ClientV2()
response = co.rerank(
    model="rerank-v4.0-pro",   
    query=query,
    documents=doc_texts,
    top_n=3,
)


for i, r in enumerate(response.results, 1):
    original_doc = docs[r.index]# this r.index is very important to map the docs retrived
    print(f"\nRank {i}")
    print(f"Index : {r.index}")
    print("Rerank Score:", r.relevance_score)
    print(original_doc.page_content[:400])
    print("-" * 100)





Rank 1
Index : 4
Rerank Score: 0.8090347
the color of their skin but by the content of their character. I have a dream . . . I have a dream that one 
day in Alabama, with its vicious racists, with its governor having his lips dripping with the words of 
interposition and nullification, one day right there in Alabama little black boys and black girls will be able
----------------------------------------------------------------------------------------------------

Rank 2
Index : 0
Rerank Score: 0.7151058
“I Have a Dream” Speech by the Rev. Martin Luther King Jr. at the “March on Washington,” 
1963 (excerpts ) 
 
I am happy to join with you today in what will go down in history as the greatest demonstration for 
freedom in the history of our nation. 
Five score years ago a great American in whose symbolic shadow we stand today signed the
----------------------------------------------------------------------------------------------------

Rank 3
Index : 1
Rerank Score: 0.7151058
“I Have a

# ___________________________________________________________
# Calling LLM

In [13]:
llm = ChatOpenAI(model="gpt-4.1-nano")
query = "What was Martin Luther King Jr. dreams about racial equality?"


docs = hybrid_retriever.invoke(query)
doc_texts = [
    doc.page_content.strip()
    for doc in docs
    if doc.page_content and doc.page_content.strip()
]


co = cohere.ClientV2()
response = co.rerank(
    model="rerank-v4.0-pro",
    query=query,
    documents=doc_texts,
    top_n=3,  # only need top 3 for answer
)


top_docs = [docs[r.index] for r in response.results] # this r.index is very important to map the docs retrived
context = "\n\n".join(doc.page_content for doc in top_docs)



prompt = f""" answer the user query based on the given context only, and if u dont know the answer say i dont have information for your query
Context:
{context}
Question: {query}
"""
response_llm = llm.invoke(prompt)
display(Markdown(response_llm.content))

Martin Luther King Jr. dreamed that in Alabama and across the nation, little black boys and black girls would be able to join together in harmony, regardless of the color of their skin, and be judged by the content of their character. He envisioned a future where racial equality and justice prevail, and where people are not judged based on race but on their character.